In [ ]:
!uname -a && cat /etc/*release

Linux c51b66b47cad 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [ ]:
!pwd
!ls -la

/content/drive/MyDrive/HPC
total 1010
-rw------- 1 root root    1565 Jun 27 14:20 matmul.c
-rw------- 1 root root    4382 Jun 27 14:20 matmul_cuda.cu
-rw------- 1 root root 1010440 Jun 27 14:20 matmul_gpu
-rw------- 1 root root   16464 Jun 27 14:20 matmul_seq


In [ ]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.00GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                3
    BogoMIPS:                4000.35
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [ ]:
%%writefile matmul.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        printf("Usage: %s <N>\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    int i, j, k;

    double (*a)[n] = malloc(sizeof(double[n][n]));
    double (*b)[n] = malloc(sizeof(double[n][n]));
    double (*c)[n] = malloc(sizeof(double[n][n]));

    if (!a || !b || !c) {
        printf("Memory allocation error for N=%d\n", n);
        return 1;
    }
    for (i = 0; i < n; i++) {
        for (j = 0; j < n; j++) {
            a[i][j] = 2.0;
            b[i][j] = 3.0;
            c[i][j] = 0.0;
        }
    }

    double start_time = omp_get_wtime();

    for (i = 0; i < n; ++i) {
        for (k = 0; k < n; k++) {
            for (j = 0; j < n; ++j) {
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }

    double run_time = omp_get_wtime() - start_time;
    printf("Computation time (N=%d): %f seconds\n", n, run_time);

    #ifdef DEBUG
    FILE *f = fopen("mat-res.txt", "w");
    if (!f) {
        perror("fopen error");
        return 1;
    }

    fprintf(f, "%d\n\n", n);

    int print_limit = (n < 1000) ? n : 1000;

    for (int i = 0; i < print_limit; i++) {
        for (int j = 0; j < print_limit; j++) {
            fprintf(f, "%.0f ", c[i][j]);
        }
        fprintf(f, "\n");
    }
    fclose(f);
    printf("Debug file 'mat-res.txt' successfully saved.\n");
    #endif
    free(a);
    free(b);
    free(c);

    return 0;
}

Overwriting matmul.c


In [ ]:
!gcc -O3 -fopenmp matmul.c -o matmul_seq



In [ ]:
!time ./matmul_seq 5000

Computation time (N=5000): 91.537347 seconds

real	1m31.851s
user	1m30.493s
sys	0m0.364s


In [ ]:
!nvidia-smi

Sat Jun 27 14:42:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
device_query_str = '''
#include <stdio.h>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);

    if (deviceCount == 0) {
        printf("Nessun dispositivo CUDA trovato.\\n");
        return 1;
    }

    for (int dev = 0; dev < deviceCount; ++dev) {
        cudaDeviceProp deviceProp;
        cudaGetDeviceProperties(&deviceProp, dev);

        printf("\\n--- Device %d: \\"%s\\" ---\\n", dev, deviceProp.name);
        printf("  CUDA Capability Major/Minor version:          %d.%d\\n", deviceProp.major, deviceProp.minor);
        printf("  Total amount of global memory:                %.2f GBytes\\n", (float)deviceProp.totalGlobalMem / (1024 * 1024 * 1024));
        printf("  Number of multiprocessors (SMs):              %d\\n", deviceProp.multiProcessorCount);
        printf("  Total amount of constant memory:              %lu bytes\\n", deviceProp.totalConstMem);
        printf("  Total amount of shared memory per block:      %lu bytes\\n", deviceProp.sharedMemPerBlock);
        printf("  Total number of registers available per block: %d\\n", deviceProp.regsPerBlock);
        printf("  Warp size:                                    %d\\n", deviceProp.warpSize);
        printf("  Maximum threads per multiprocessor (SM):      %d\\n", deviceProp.maxThreadsPerMultiProcessor);
        printf("  Maximum threads per block:                    %d\\n", deviceProp.maxThreadsPerBlock);
        printf("  Max dimension size of a thread block (x,y,z): %d, %d, %d\\n", deviceProp.maxThreadsDim[0], deviceProp.maxThreadsDim[1], deviceProp.maxThreadsDim[2]);
        printf("  Max dimension size of a grid size    (x,y,z): %d, %d, %d\\n", deviceProp.maxGridSize[0], deviceProp.maxGridSize[1], deviceProp.maxGridSize[2]);
    }
    return 0;
}
'''
with open('deviceQuery.cu', 'w') as f:
    f.write(device_query_str)

In [ ]:
!nvcc -arch=sm_75 deviceQuery.cu -o deviceQuery && ./deviceQuery


--- Device 0: "Tesla T4" ---
  CUDA Capability Major/Minor version:          7.5
  Total amount of global memory:                14.56 GBytes
  Number of multiprocessors (SMs):              40
  Total amount of constant memory:              65536 bytes
  Total amount of shared memory per block:      49152 bytes
  Total number of registers available per block: 65536
  Warp size:                                    32
  Maximum threads per multiprocessor (SM):      1024
  Maximum threads per block:                    1024
  Max dimension size of a thread block (x,y,z): 1024, 1024, 64
  Max dimension size of a grid size    (x,y,z): 2147483647, 65535, 65535


In [10]:
%%writefile matmul_cuda.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// ---------------------------------------------------------
// GPU KERNEL: Executed on the graphics card
// ---------------------------------------------------------
__global__ void matMulKernel(double *a, double *b, double *c, int n) {
    // 1. Calculate the global 2D coordinates of the thread
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // 2. Check that the thread is within the matrix boundaries
    if (row < n && col < n) {
        double sum = 0.0;

        // 3. The actual computation (The only remaining loop!)
        // Notice how we access the 1D array simulating a 2D matrix (row * n + col)
        for (int k = 0; k < n; k++) {
            sum += a[row * n + k] * b[k * n + col];
        }

        // 4. Save the result
        c[row * n + col] = sum;
    }
}

// ---------------------------------------------------------
// MAIN HOST: Executed on the CPU
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        return 1;
    }
    int n = atoi(argv[1]);
    size_t size = (size_t)n * n * sizeof(double);

    // --- HOST MEMORY ALLOCATION (CPU) ---
    // I use "flattened" 1D arrays because they are the safest for GPU transfers
    double *h_a = (double *)malloc(size);
    double *h_b = (double *)malloc(size);
    double *h_c = (double *)malloc(size);

    if (!h_a || !h_b || !h_c) {
        printf("Host RAM allocation error for N=%d\n", n);
        return 1;
    }

    // Data initialization on the CPU
    for (int i = 0; i < n * n; i++) {
        h_a[i] = 2.0;
        h_b[i] = 3.0;
        h_c[i] = 0.0;
    }

    // --- PREPARATION OF CUDA TIMERS ---
    cudaEvent_t start, stop_h2d, stop_kernel, stop_d2h;
    cudaEventCreate(&start);
    cudaEventCreate(&stop_h2d);
    cudaEventCreate(&stop_kernel);
    cudaEventCreate(&stop_d2h);

    // --- DEVICE MEMORY ALLOCATION (GPU) ---
    double *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // =======================================================
    // 1. DATA TRANSFER: CPU -> GPU (HostToDevice)
    // =======================================================
    cudaEventRecord(start); // Start the timer

    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    cudaEventRecord(stop_h2d); // Record end of H2D

    // =======================================================
    // 2. KERNEL EXECUTION
    // =======================================================
    // Define the magic block in multiples of 32 (256 total threads)
    dim3 threadsPerBlock(16, 16);
    // Calculate the grid by rounding up to cover the entire matrix
    dim3 numBlocks((n + 15) / 16, (n + 15) / 16);

    matMulKernel<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stop_kernel); // Record end of GPU computation

    // =======================================================
    // 3. RESULT TRANSFER: GPU -> CPU (DeviceToHost)
    // =======================================================
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop_d2h); // Record end of D2H
    cudaEventSynchronize(stop_d2h); // Wait for everything to completely finish

    // --- CALCULATION AND PRINTING OF TIMES ---
    float ms_h2d, ms_kernel, ms_d2h;
    cudaEventElapsedTime(&ms_h2d, start, stop_h2d);
    cudaEventElapsedTime(&ms_kernel, stop_h2d, stop_kernel);
    cudaEventElapsedTime(&ms_d2h, stop_kernel, stop_d2h);

    printf("Matrix Dimension N           : %d\n", n);
    printf("Transfer Time CPU -> GPU     : %f ms\n", ms_h2d);
    printf("GPU Kernel Computation Time  : %f ms\n", ms_kernel);
    printf("Transfer Time GPU -> CPU     : %f ms\n", ms_d2h);
    printf("Total Time (Data + Compute)  : %f ms\n", ms_h2d + ms_kernel + ms_d2h);

    // --- MEMORY CLEANUP ---
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    // Destroy the timers
    cudaEventDestroy(start); cudaEventDestroy(stop_h2d);
    cudaEventDestroy(stop_kernel); cudaEventDestroy(stop_d2h);

    return 0;
}

Overwriting matmul_cuda.cu


In [11]:
!nvcc -arch=sm_75 -O3 matmul_cuda.cu -o matmul_gpu

In [12]:
!/content/matmul_gpu 10000

Matrix Dimension N           : 10000
Transfer Time CPU -> GPU     : 346.830353 ms
GPU Kernel Computation Time  : 9253.874023 ms
Transfer Time GPU -> CPU     : 167.804352 ms
Total Time (Data + Compute)  : 9768.508789 ms


In [13]:
!nvprof /content/matmul_gpu 10000

==1986== NVPROF is profiling process 1986, command: /content/matmul_gpu 10000
Matrix Dimension N           : 10000
Transfer Time CPU -> GPU     : 349.782318 ms
GPU Kernel Computation Time  : 9327.497070 ms
Transfer Time GPU -> CPU     : 190.541885 ms
Total Time (Data + Compute)  : 9867.821289 ms
==1986== Profiling application: /content/matmul_gpu 10000
==1986== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   94.53%  9.32721s         1  9.32721s  9.32721s  9.32721s  matMulKernel(double*, double*, double*, int)
                    3.54%  349.31ms         2  174.66ms  171.58ms  177.73ms  [CUDA memcpy HtoD]
                    1.93%  190.13ms         1  190.13ms  190.13ms  190.13ms  [CUDA memcpy DtoH]
      API calls:   98.69%  9.86742s         3  3.28914s  171.75ms  9.51771s  cudaMemcpy
                    1.23%  123.29ms         4  30.821ms     452ns  123.28ms  cudaEventCreate
                    0.06%  5.6257ms      

In [1]:
%%writefile matmul_cuda_tiled.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Permettiamo allo script Bash di decidere il TILE_SIZE durante la compilazione.
// Se non viene specificato dal compilatore, usa 16 di default.
#ifndef TILE_SIZE
#define TILE_SIZE 16
#endif

// ---------------------------------------------------------
// GPU KERNEL (TILED VERSION)
// ---------------------------------------------------------
__global__ void matMulKernel(double *a, double *b, double *c, int n) {
    __shared__ double As[TILE_SIZE][TILE_SIZE];
    __shared__ double Bs[TILE_SIZE][TILE_SIZE];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + tx;
    int row = blockIdx.y * TILE_SIZE + ty;

    double sum = 0.0;

    for (int t = 0; t < (n + TILE_SIZE - 1) / TILE_SIZE; t++) {
        if (row < n && t * TILE_SIZE + tx < n)
            As[ty][tx] = a[row * n + t * TILE_SIZE + tx];
        else
            As[ty][tx] = 0.0;

        if (t * TILE_SIZE + ty < n && col < n)
            Bs[ty][tx] = b[(t * TILE_SIZE + ty) * n + col];
        else
            Bs[ty][tx] = 0.0;

        __syncthreads();

        for (int k = 0; k < TILE_SIZE; k++) {
            sum += As[ty][k] * Bs[k][tx];
        }
        __syncthreads();
    }

    if (row < n && col < n) {
        c[row * n + col] = sum;
    }
}

// ---------------------------------------------------------
// MAIN HOST
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Error: specify the dimension N.\n");
        return 1;
    }
    int n = atoi(argv[1]);
    size_t size = (size_t)n * n * sizeof(double);

    double *h_a = (double *)malloc(size);
    double *h_b = (double *)malloc(size);
    double *h_c = (double *)malloc(size);

    for (int i = 0; i < n * n; i++) {
        h_a[i] = 2.0; h_b[i] = 3.0; h_c[i] = 0.0;
    }

    cudaEvent_t start, stop_h2d, stop_kernel, stop_d2h;
    cudaEventCreate(&start); cudaEventCreate(&stop_h2d);
    cudaEventCreate(&stop_kernel); cudaEventCreate(&stop_d2h);

    double *d_a, *d_b, *d_c;
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    cudaEventRecord(start);
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaEventRecord(stop_h2d);

    // Usiamo il TILE_SIZE per definire la dimensione del blocco e della griglia
    dim3 threadsPerBlock(TILE_SIZE, TILE_SIZE);
    dim3 numBlocks((n + TILE_SIZE - 1) / TILE_SIZE, (n + TILE_SIZE - 1) / TILE_SIZE);

    matMulKernel<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaEventRecord(stop_kernel);

    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    cudaEventRecord(stop_d2h);
    cudaEventSynchronize(stop_d2h);

    float ms_h2d, ms_kernel, ms_d2h;
    cudaEventElapsedTime(&ms_h2d, start, stop_h2d);
    cudaEventElapsedTime(&ms_kernel, stop_h2d, stop_kernel);
    cudaEventElapsedTime(&ms_d2h, stop_kernel, stop_d2h);

    // Stampa per il grep dello script bash
    printf("BlockSize: %dx%d\n", TILE_SIZE, TILE_SIZE);
    printf("H2D Time : %f ms\n", ms_h2d);
    printf("Kernel Time : %f ms\n", ms_kernel);
    printf("D2H Time : %f ms\n", ms_d2h);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    cudaEventDestroy(start); cudaEventDestroy(stop_h2d);
    cudaEventDestroy(stop_kernel); cudaEventDestroy(stop_d2h);

    return 0;
}

Writing matmul_cuda_tiled.cu


In [2]:
!nvcc -arch=sm_75 -O3 matmul_cuda_tiled.cu -o matmul_gpu_tiled

In [3]:
!/content/matmul_gpu_tiled 10000

BlockSize: 16x16
H2D Time : 177.935837 ms
Kernel Time : 8509.871094 ms
D2H Time : 179.956345 ms


In [18]:
!nvprof /content/matmul_gpu_tiled 10000

==2338== NVPROF is profiling process 2338, command: /content/matmul_gpu_tiled 10000
Matrix Dimension N           : 10000
Transfer Time CPU -> GPU     : 348.350616 ms
GPU Kernel Computation Time  : 8849.871094 ms
Transfer Time GPU -> CPU     : 176.936005 ms
Total Time (Data + Compute)  : 9375.157227 ms
==2338== Profiling application: /content/matmul_gpu_tiled 10000
==2338== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   94.41%  8.84959s         1  8.84959s  8.84959s  8.84959s  matMulKernel(double*, double*, double*, int)
                    3.71%  347.86ms         2  173.93ms  170.85ms  177.01ms  [CUDA memcpy HtoD]
                    1.88%  176.52ms         1  176.52ms  176.52ms  176.52ms  [CUDA memcpy DtoH]
      API calls:   98.65%  9.37476s         3  3.12492s  171.02ms  9.02650s  cudaMemcpy
                    1.27%  121.00ms         4  30.251ms     486ns  121.00ms  cudaEventCreate
                    0.05%  5.

In [19]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/HPC')

Mounted at /content/drive


In [ ]:
!find / -type f -name nsys 2>/dev/null

/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys


In [23]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys profile -t cuda,osrt -o /content/drive/MyDrive/HPC/report_matmul --stats=true --force-overwrite=true /content/matmul_gpu 10000

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
Matrix Dimension N           : 10000
Transfer Time CPU -> GPU     : 338.762482 ms
GPU Kernel Computation Time  : 9788.414062 ms
Transfer Time GPU -> CPU     : 185.852966 ms
Total Time (Data + Compute)  : 10313.029297 ms
Generating '/tmp/nsys-report-95dd.qdstrm'
[1/7] [========================100%] report_matmul.nsys-rep
[2/7] [========================100%] report_matmul.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  ----------------------
     98.3   10,907,410,452        118  92,435,681.8  100,136,569.5     2,073  281,402,599  32,626,258.4  poll                  
      1.6      179,045,539        535     3

In [24]:
!/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/nsys profile -t cuda,osrt -o /content/drive/MyDrive/HPC/report_matmul_tiled --stats=true --force-overwrite=true /content/matmul_gpu_tiled 10000

Invalid plugin configuration: Executable path does not exist: /opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64/plugins/efa_metrics/nic_sampler
Matrix Dimension N           : 10000
Transfer Time CPU -> GPU     : 345.281677 ms
GPU Kernel Computation Time  : 9154.311523 ms
Transfer Time GPU -> CPU     : 175.081757 ms
Total Time (Data + Compute)  : 9674.674805 ms
Generating '/tmp/nsys-report-fdfb.qdstrm'
[1/7] [========================100%] report_matmul_tiled.nsys-rep
[2/7] [========================100%] report_matmul_tiled.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  ----------------------
     98.3   10,217,053,165        111  92,045,524.0  100,127,417.0     1,598  295,072,206  34,267,165.7  poll                  
      1.7      174,773,221      

In [ ]:
#!/usr/bin/env bash

# Exit immediately if any command fails
set -euo pipefail

RESULTS_DIR="./results"
mkdir -p "$RESULTS_DIR"

CSV_FILE="$RESULTS_DIR/cuda_scalability.csv"
RAW_LOG="$RESULTS_DIR/cuda_raw.log"

# Creiamo l'intestazione del CSV
# Aggiungiamo anche i tempi di trasferimento per un'analisi approfondita!
echo "N,BlockSize_1D,ThreadsPerBlock,H2D_ms,Kernel_ms,D2H_ms" > "$CSV_FILE"
> "$RAW_LOG"

echo "=================================================="
echo " INIZIO BENCHMARK CUDA"
echo "=================================================="

# Scegliamo dimensioni (es. 2000x2000, 4000x4000, 8000x8000)
# Assicurati che l'N più grande su CPU superi i 30 secondi richiesti!
SIZES=(5000 10000 15000)

# Configurazioni dei Blocchi 2D (TILE_SIZE).
# 8 significa blocco 8x8 (64 thread)
# 16 significa blocco 16x16 (256 thread) - Solitamente lo "sweet spot"
# 32 significa blocco 32x32 (1024 thread) - Il limite hardware massimo per SM
BLOCK_SIZES=(8 16 32)

for N in "${SIZES[@]}"; do
    echo ""
    echo "--- Test Dimensione Matrice N = $N ---"

    for B in "${BLOCK_SIZES[@]}"; do
        TOTAL_THREADS=$((B * B))
        printf "Compilando ed Eseguendo Blocco %2dx%-2d (%4d threads) ... " "$B" "$B" "$TOTAL_THREADS"

        # 1. Compiliamo CUDA passando la dimensione del blocco come macro!
        # -DTILE_SIZE=$B sostituirà il valore nel codice sorgente .cu
        nvcc matmul.cu -DTILE_SIZE=$B -O3 -o matmul_gpu_test

        # 2. Eseguiamo
        OUTPUT=$(./matmul_gpu_test "$N")

        # Salviamo l'output completo per debug
        echo "=== N=$N, Block=${B}x${B} ===" >> "$RAW_LOG"
        echo "$OUTPUT" >> "$RAW_LOG"

        # 3. Estraiamo i tempi con grep e awk
        TIME_H2D=$(echo "$OUTPUT" | grep "H2D Time" | awk '{print $(NF-1)}')
        TIME_KER=$(echo "$OUTPUT" | grep "Kernel Time" | awk '{print $(NF-1)}')
        TIME_D2H=$(echo "$OUTPUT" | grep "D2H Time" | awk '{print $(NF-1)}')

        # Stampa a video
        echo "Kernel: ${TIME_KER} ms"

        # 4. Scriviamo nel CSV
        echo "$N,$B,$TOTAL_THREADS,$TIME_H2D,$TIME_KER,$TIME_D2H" >> "$CSV_FILE"
    done
done

echo "=================================================="
echo " BENCHMARK COMPLETATO!"
echo " Risultati salvati in: $CSV_FILE"